# 📘 Notebook 2f: Remote Development from VS Code

## Overview

Data scientists on the customer's team likely live in VS Code / PyCharm / Cursor — not Snowsight. This notebook is the best-practice reference for **how a team connects their laptop to Snowflake for real work**: local editing, git-tracked code, cloud execution.

The scope is narrow on purpose: *how do I set up my laptop once, and what does my daily loop look like?*

### What's Covered

| Section | Topic |
|---|---|
| 1 | Why develop locally (what you gain over Snowsight-only) |
| 2 | Connection best practices — `connections.toml`, key-pair auth, one file per env |
| 3 | VS Code extensions worth installing |
| 4 | Recommended repo layout + the local dev loop |
| 5 | Code samples — session, SQL, ML Job, model registry, feature store |
| 6 | Gotchas + security do's and don'ts |

### When you'd use this

- Training scripts that live in git and run in CI
- Data-science workflows where SQL + Python move together (dbt + Snowpark)
- Any code that ends up in Airflow — develop locally, submit via `submit_file()` (see `02_e`)

### References

- [Snowflake VS Code extension](https://docs.snowflake.com/en/user-guide/vscode-ext)
- [Python connector: `connections.toml`](https://docs.snowflake.com/en/developer-guide/python-connector/python-connector-connect#connecting-using-the-connections-toml-file)
- [Snowpark Session — named connections](https://docs.snowflake.com/en/developer-guide/snowpark/python/creating-session#connecting-with-the-connections-toml-file)
- [Key-pair authentication](https://docs.snowflake.com/en/user-guide/key-pair-auth)


## 📘 Section 1 — Why develop locally

Snowsight notebooks are great for exploration. They're less great for:

| Concern | Snowsight | VS Code + `submit_file()` |
|---|---|---|
| **Git history of your training code** | Notebook JSON, messy diffs | Plain `.py`, normal `git diff` |
| **Local linting + type-checking** | None | `ruff`, `mypy`, IDE intellisense |
| **Debugger breakpoints** | Limited | Full VS Code debugger (on the parts that don't need SPCS) |
| **Unit tests** | Awkward to run | `pytest` next to the code |
| **Multi-file refactors** | One file at a time | Rename/refactor across the repo |
| **Code review** | Hard | Standard PR flow |

What you **don't** give up by going local:

- GPU access — `submit_file(..., compute_pool="WAFER_TRAINING_POOL")` puts the heavy work on SPCS
- The data — still stays in Snowflake; you query it by reference
- Access control — same Snowflake RBAC, just accessed via your user/service user

The mental model: *edit locally, execute remotely.*


## 📘 Section 2 — Connection best practices

Snowflake's recommended connection config is `~/.snowflake/connections.toml`. **Every Snowflake client library reads it** (Python connector, Snowpark, Snowflake CLI, VS Code extension). Configure once, reuse everywhere.

### 2.1 Create the file (once per machine)

```bash
mkdir -p ~/.snowflake
touch ~/.snowflake/connections.toml
chmod 600 ~/.snowflake/connections.toml
```

### 2.2 Generate a key-pair (one-time per user)

```bash
# Generate an unencrypted key for dev (use encrypted for prod / shared machines)
openssl genrsa 2048 | openssl pkcs8 -topk8 -inform PEM -out ~/.snowflake/keys/rsa_key.p8 -nocrypt
openssl rsa -in ~/.snowflake/keys/rsa_key.p8 -pubout -out ~/.snowflake/keys/rsa_key.pub
chmod 600 ~/.snowflake/keys/rsa_key.p8

# Register the public key on your Snowflake user
#   ALTER USER HMASSA SET RSA_PUBLIC_KEY='<paste contents of rsa_key.pub, no headers>';
```

**Why key-pair over password:**
- No rotating-password pain, no password in any config file
- Works unattended (CI, scheduled scripts, Airflow workers)
- Required for service users in most enterprise policies

### 2.3 Populate `connections.toml`

One section per environment. Name the default; everything else is opt-in.

```toml
default_connection_name = "dev"

[dev]
account   = "sfsenorthamerica-hmassa_aws1"
user      = "HMASSA"
authenticator     = "SNOWFLAKE_JWT"
private_key_file  = "/Users/hmassa/.snowflake/keys/rsa_key.p8"
role      = "ACCOUNTADMIN"
warehouse = "WAFER_DEMO_WH"
database  = "WAFER_YIELD_DEMO"
schema    = "ML_MODELS"

[prod]
account   = "sfsenorthamerica-hmassa_aws1"
user      = "ML_SERVICE_USER"
authenticator    = "SNOWFLAKE_JWT"
private_key_file = "/Users/hmassa/.snowflake/keys/prod_service.p8"
role      = "ML_SERVICE_ROLE"
warehouse = "ML_PROD_WH"
database  = "WAFER_YIELD_DEMO"
schema    = "ML_MODELS"
```

### 2.4 Use the named connection — Python, SQL, CLI, all the same

```python
# Snowpark
from snowflake.snowpark import Session
session = Session.builder.config("connection_name", "dev").create()
```

```python
# Python connector
import snowflake.connector
conn = snowflake.connector.connect(connection_name="dev")
```

```bash
# Snowflake CLI (use `cortex connections` or `snow connection`)
cortex connections set dev
cortex search object "WAFER"
```

### 2.5 One file, many environments, zero secrets in code

| Rule | Why |
|---|---|
| Never hardcode account/user/password in a `.py` file | Leaks in git history |
| Never commit `connections.toml` or `.p8` keys | Use `.gitignore` to block `*.toml` and `*.p8` in the repo |
| Use different `user` + `role` per env (dev/staging/prod) | Prevents a dev mistake from touching prod |
| Rotate keys on a schedule (`ALTER USER ... SET RSA_PUBLIC_KEY_2 = ...`) | Two-slot rotation — no downtime |
| For CI/CD, store the private key in the secrets manager and mount it, don't bake it into the image | Standard 12-factor |


## 📘 Section 3 — VS Code extensions

Three that every Snowflake dev wants installed:

| Extension | What it gives you |
|---|---|
| **Snowflake** ([marketplace](https://marketplace.visualstudio.com/items?itemName=Snowflake.snowflake-vsc)) | Object explorer for DBs/schemas/tables/models/services. Run `.sql` files against your named connection. Preview query results inline. Authored by Snowflake. |
| **Python** (Microsoft) | Debugger, IntelliSense, test runner. Not optional. |
| **Jupyter** (Microsoft) | Run `.ipynb` files locally against your Snowpark session. Good middle ground between Snowsight and full `.py` scripts. |

Optional but common:

- **Ruff** — fast linter/formatter. Snowflake's own ML codebases use it.
- **Even Better TOML** — syntax highlighting for `connections.toml` + `pyproject.toml`.
- **GitHub Copilot** / **Cursor** — the team probably already has one of these.

The Snowflake extension specifically picks up `~/.snowflake/connections.toml` automatically — no separate config.


## 📘 Section 4 — Repo layout + dev loop

### Recommended repo layout

```
wafer-ml/
├── .gitignore                  # excludes *.toml, *.p8, .venv, __pycache__
├── pyproject.toml              # deps pinned here (snowflake-ml-python>=1.9.0, torch, etc.)
├── .env.example                # documents env vars, but no real values
├── ml/
│   ├── train_wafer_model.py    # the ML Job — runs on SPCS
│   ├── features.py             # pure-Python helpers (unit-testable locally)
│   └── tests/
│       └── test_features.py    # pytest — runs on every PR
├── sql/
│   └── setup.sql               # warehouse / role / schema bootstrap
├── dags/
│   └── wafer_yield_pipeline.py # Airflow DAG (references ml/train_wafer_model.py)
├── submit_training.py          # dev helper — run this from VS Code to submit an ad-hoc job
└── README.md
```

### Daily dev loop

1. **Edit** `ml/train_wafer_model.py` in VS Code. IntelliSense resolves `snowflake.ml` APIs via the venv.
2. **Test pure-Python locally** — `pytest ml/tests/` for anything that doesn't need Snowflake (feature transforms, parsing, preprocessors).
3. **Sanity-check against real data** — run `python submit_training.py` for a scoped dev job against `v1_train`. Logs stream back; iterate.
4. **Commit + push**. CI lints + tests. Airflow's next scheduled run picks up the new code automatically (if your DAG reads from the same repo path).
5. **Promote** via alias or tag — the same Airflow DAG's `promote` task handles this based on held-out metrics.

The key property: **the same file runs in three places — your laptop, an Airflow task, a Snowflake notebook — without modification.**


## 📘 Section 5 — Code samples

Everything below is what you'd have open in a VS Code tab. All of it uses the `dev` connection from `connections.toml` — no credentials in the code.


In [ ]:
# ========================================================================
# Creating a Snowpark session from a named connection
# ========================================================================
from snowflake.snowpark import Session

session = Session.builder.config("connection_name", "dev").create()

# BEST PRACTICE — always pin database + schema after creating the session.
# ML Jobs (@remote / submit_file) will fail to register without an active DB.
session.sql("USE DATABASE WAFER_YIELD_DEMO").collect()
session.sql("USE SCHEMA ML_MODELS").collect()

print("account :", session.sql("SELECT CURRENT_ACCOUNT()").collect()[0][0])
print("user    :", session.sql("SELECT CURRENT_USER()").collect()[0][0])
print("role    :", session.sql("SELECT CURRENT_ROLE()").collect()[0][0])
print("wh      :", session.get_current_warehouse())
print("db/sch  :", session.get_current_database(), "/", session.get_current_schema())


In [ ]:
# ========================================================================
# ONE-TIME SETUP — idempotent, safe to re-run
# ========================================================================
# The ML_JOB_STAGE holds the submitted payload + intermediate artifacts.
# Create it once; everything else (@remote / submit_file) references it by name.
session.sql("""
    CREATE STAGE IF NOT EXISTS WAFER_YIELD_DEMO.ML_MODELS.ML_JOB_STAGE
        ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE')
        DIRECTORY = (ENABLE = TRUE)
        COMMENT = 'Payload + results for ML Jobs (VS Code submits here)'
""").collect()
print("ML_JOB_STAGE ready")


In [ ]:
# ========================================================================
# Run a query, pull to pandas — the simplest local workflow
# ========================================================================
# Data lives in Snowflake. The query runs in Snowflake. You only materialize
# what you need locally.
df = session.sql('''
    SELECT "WAFER_ID", "output_feature_0" AS yield_prob
    FROM WAFER_YIELD_DEMO.INFERENCE.WAFER_YIELD_PREDICTIONS
    WHERE "RUN_ID" = (SELECT MAX("RUN_ID") FROM WAFER_YIELD_DEMO.INFERENCE.WAFER_YIELD_PREDICTIONS)
    LIMIT 1000
''').to_pandas()

print(df.shape)
print(df.head())


In [ ]:
# ========================================================================
# submit_training.py — laptop-to-Snowflake ML Job submission
# ========================================================================
# Edit ml/train_wafer_model.py in VS Code, then run this. Confirmed working
# against haleyconnect (V_DEV_*** jobs register successfully).
from datetime import datetime
from snowflake.snowpark import Session
from snowflake.ml.jobs import submit_file

session = Session.builder.config("connection_name", "dev").create()

# BEST PRACTICE — set DB + schema before submitting. ML Jobs will otherwise
# error with "Database must be specified either in the session context or as a parameter."
session.sql("USE DATABASE WAFER_YIELD_DEMO").collect()
session.sql("USE SCHEMA ML_MODELS").collect()

version = f"V_DEV_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
job = submit_file(
    "ml/train_wafer_model.py",
    compute_pool="WAFER_TRAINING_POOL",
    stage_name="@WAFER_YIELD_DEMO.ML_MODELS.ML_JOB_STAGE",
    env_vars={
        "DATASET_VERSION":      "v1_train",
        "DATASET_TEST_VERSION": "v1_test",
        "NEW_VERSION_NAME":     version,
    },
)
print("Submitted:", job.id)
job.wait()
print("Status:", job.status)
print(job.get_logs())


In [ ]:
# ========================================================================
# Registering a model from your laptop
# ========================================================================
# You usually don't do this from local — you let the ML Job register inside
# SPCS so lineage tracks to the job. But for a one-off (e.g. migrating a
# pickle), the exact same API works locally.
from snowflake.ml.registry import Registry
import pandas as pd, joblib

reg = Registry(session, database_name="WAFER_YIELD_DEMO", schema_name="ML_MODELS")

# Example — if you had a local pickled sklearn/xgboost model
# model = joblib.load("my_model.pkl")
# sample = pd.read_parquet("sample_input.parquet").head(1).astype("float32")
# mv = reg.log_model(
#     model=model,
#     model_name="WAFER_YIELD_CHAMPION",
#     version_name="V_LOCAL_TEST",
#     sample_input_data=sample,
#     metrics={"accuracy": 0.78},
# )

# What you CAN always do: inspect what's already registered
print(reg.show_models())


In [ ]:
# ========================================================================
# Reading from the Feature Store from your laptop
# ========================================================================
# Feature views are SQL under the hood — your VS Code session can query them
# exactly like any other table.
from snowflake.ml.feature_store import FeatureStore, CreationMode

fs = FeatureStore(
    session=session,
    database="WAFER_YIELD_DEMO",
    name="FEATURES",
    default_warehouse="WAFER_DEMO_WH",
    creation_mode=CreationMode.FAIL_IF_NOT_EXIST,
)

try:
    fv = fs.get_feature_view("WAFER_YIELD_FEATURES", "1.0")
    print("Feature view columns:", fv.feature_df.columns)
except Exception as e:
    print("Feature view not found (you may need to run Notebook 01 first):", e)


## 📘 Section 5b — Verified end-to-end smoke test

This cell is the **minimum viable ML Job from VS Code** — three imports, a decorated function, submit, wait, get logs. If this works on your laptop, the full `submit_file()` path will too. Confirmed running against `WAFER_TRAINING_POOL` in ~30 seconds.


In [ ]:
# ========================================================================
# Minimum viable ML Job — hello world from your laptop
# ========================================================================
# Uses the `session` created in the cell above. Run after the setup cell.
from snowflake.ml.jobs import remote

@remote("WAFER_TRAINING_POOL",
        stage_name="@WAFER_YIELD_DEMO.ML_MODELS.ML_JOB_STAGE")
def hello_from_spcs(name: str = "VS Code"):
    from datetime import datetime
    print(f"{datetime.now()}  Hello {name} from the ML Job!")

job = hello_from_spcs("customer demo")
print("Submitted:", job.id)
job.wait()
print("Status:", job.status)
print("--- logs ---")
print(job.get_logs())


## 📘 Section 6 — Gotchas & security checklist

### Things that trip up new teams

| Gotcha | Fix |
|---|---|
| **`ValueError: Database must be specified ...` on `@remote` / `submit_file`** | `USE DATABASE` + `USE SCHEMA` right after creating the session — ML Jobs need a DB context to register the job definition |
| Python 3.11 in local venv, but ML Jobs require 3.10 | `python3.10 -m venv .venv` — the Runtime Job API currently pins 3.10 |
| `Session.builder.create()` with no args — uses whatever env vars happen to be set | Always pass `.config("connection_name", "<name>")` explicitly |
| Committed `connections.toml` by accident | `.gitignore`: `*.toml`, `**/keys/`, `.snowflake/` |
| Training script works locally, fails in ML Job | Usually a local-only dep (matplotlib GUI backend, absolute file paths). Use `submit_directory` with a proper `requirements.txt` |
| Role mismatch — laptop runs as `ACCOUNTADMIN`, Airflow runs as `ML_SERVICE_ROLE` | Use separate connections per env and test with the service role locally before shipping |
| SPCS compute pool is suspended on first call — long wait | `ALTER COMPUTE POOL ... RESUME;` or set `auto_resume = TRUE` on the pool |
| Stage doesn't exist: `@WAFER_YIELD_DEMO.ML_MODELS.ML_JOB_STAGE` | Run the one-time setup cell (Section 5) once per env — `CREATE STAGE IF NOT EXISTS ...` is idempotent |

### Security do's

- ✅ Key-pair auth for both humans and service users
- ✅ Separate role per environment (dev/staging/prod) with least privilege
- ✅ Rotate keys via `RSA_PUBLIC_KEY_2` before revoking `RSA_PUBLIC_KEY` — no downtime
- ✅ Store prod private keys in a secrets manager, mount at runtime
- ✅ Enable network policy on service users (IP allow-list)
- ✅ Enable MFA for user accounts, not for service accounts

### Security don'ts

- ❌ Never commit `.p8`, `.pem`, or `connections.toml`
- ❌ Never put passwords in `.env` files that live in git
- ❌ Never share a personal user with an automated pipeline — always use a dedicated service user
- ❌ Never grant `ACCOUNTADMIN` to a pipeline role — custom roles only


## 📘 Summary

> **Every Snowflake client reads `~/.snowflake/connections.toml`. Set it up once with key-pair auth, use named connections everywhere, keep your training code in git, submit to SPCS via `submit_file()` when you need cloud compute.**

### What to hand the customer's ML team

1. The `connections.toml` template from Section 2
2. The repo layout from Section 4
3. `submit_training.py` from Section 5 — it's their entry point for everything
4. The gotchas table from Section 6 — saves them a week of trial and error
5. The Snowflake VS Code extension

### Related notebooks

- `02_e_ML_Jobs_and_Airflow` — what this VS Code code eventually becomes in production (Airflow DAG)
- `02_GPU_Training_Wafers` — the in-Snowsight version of the same model
